## Experiment 5: Correct CUDA Benchmarking and Synchronization



Objective

The objective of this experiment is to develop a reliable benchmarking procedure for CUDA kernels and understand how CUDA's asynchronous execution model affects performance measurements.

CUDA kernel launches normally return control to the CPU before GPU execution has necessarily completed. As a result, conventional CPU timing code can significantly underestimate kernel execution time.

A benchmarking helper will therefore be implemented using:

* Warm-up iterations.
* Multiple measured iterations.
* Explicit CUDA synchronization.
* Average execution time across repeated runs.

A representative timing structure will ensure that GPU work completes before elapsed time is recorded.

The experiment will focus on:

* The difference between asynchronous kernel launch time and actual execution time.
* The importance of `torch.cuda.synchronize()`.
* The effect of warm-up iterations.
* Variability between individual measurements.
* The importance of repeated measurements when comparing CUDA implementations.

The benchmarking approach will then be used to compare kernels implemented throughout the lecture.

The purpose of this experiment is to establish a correct and repeatable CUDA benchmarking methodology so that later performance conclusions are based on actual GPU execution rather than misleading host-side timing.


## Implementation

In [1]:
!nvidia-smi


Sat Sep 12 02:48:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             30W /   70W |     667MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ninja


In [3]:
#importing libraries
import torch # standard torch operations
from torch.utils.cpp_extension import load_inline # loading inline C++/CUDA code
import pandas as pd # data manipulation
import time # timing operations
from matplotlib import pyplot as plt # plotting

print("Library Versions:")
print(f"torch: {torch.__version__}")
print(f"pandas: {pd.__version__}")


Library Versions:
torch: 2.11.0+cu128
pandas: 2.2.3


Complete the previous experiment to ensure that we use the previous scripts here for benchmarking

In [5]:
python_benchmarking_script = """

import torch
import argparse
import time
from torch.utils.cpp_extension import load_inline


parser = argparse.ArgumentParser(
    description="Matrix Multiplication Benchmark"
)

parser.add_argument(
    "--input_size",
    type=int,
    default=512,
    help="Input size of square matrices"
)

parser.add_argument(
    "--kernel",
    type=str,
    choices=["naive", "tiled", "torch"],
    default="naive",
    help="Kernel to benchmark"
)

parser.add_argument(
    "--warmup",
    type=int,
    default=10,
    help="Number of warm-up iterations"
)

parser.add_argument(
    "--iters",
    type=int,
    default=100,
    help="Number of timed iterations"
)

args = parser.parse_args()


def read_file(filename):
    with open(filename, "r") as f:
        return f.read()


cpp_matmul = read_file("./matmul.cpp")
cuda_matmul = read_file("./matmul_cuda.cu")


matmul_module = load_inline(
    name="matmul_v2",
    cpp_sources=cpp_matmul,
    cuda_sources=cuda_matmul,
    functions=["naive_cuda", "tiled_cuda"],
    extra_cuda_cflags=["-O3"],
    extra_cflags=["-O3"],
    verbose=False
)


N = args.input_size


A = torch.randn(
    N,
    N,
    device="cuda",
    dtype=torch.float32
)

B = torch.randn(
    N,
    N,
    device="cuda",
    dtype=torch.float32
)


# --------------------------------------------------
# Helper: run selected kernel
# --------------------------------------------------

def run_kernel():

    if args.kernel == "naive":
        return matmul_module.naive_cuda(A, B)

    elif args.kernel == "tiled":
        return matmul_module.tiled_cuda(A, B)

    elif args.kernel == "torch":
        return A @ B


# --------------------------------------------------
# Warm-up
# --------------------------------------------------

for _ in range(args.warmup):
    C = run_kernel()

torch.cuda.synchronize()


# --------------------------------------------------
# Correct synchronized benchmark
# --------------------------------------------------

start = time.perf_counter()

for _ in range(args.iters):
    C = run_kernel()

torch.cuda.synchronize()

end = time.perf_counter()


total_time = end - start
avg_time_s = total_time / args.iters
avg_time_ms = avg_time_s * 1000


print(
    f"kernel={args.kernel}, "
    f"matrix_size={N}x{N}, "
    f"warmup={args.warmup}, "
    f"iters={args.iters}, "
    f"avg_latency_ms={avg_time_ms:.6f}"
)

"""

In [6]:
def write_script_to_file(script, filename):
    with open(filename, 'w') as f:
        f.write(script)



In [9]:
write_script_to_file(python_benchmarking_script, 'benchmark_matmul.py')

In [10]:
!python benchmark_matmul.py --kernel naive --input_size 128
!python benchmark_matmul.py --kernel tiled --input_size 128
!python benchmark_matmul.py --kernel torch --input_size 128

!python benchmark_matmul.py --kernel naive --input_size 512
!python benchmark_matmul.py --kernel tiled --input_size 512
!python benchmark_matmul.py --kernel torch --input_size 512

!python benchmark_matmul.py --kernel naive --input_size 1024
!python benchmark_matmul.py --kernel tiled --input_size 1024
!python benchmark_matmul.py --kernel torch --input_size 1024

kernel=naive, matrix_size=128x128, warmup=10, iters=100, avg_latency_ms=0.025154
kernel=tiled, matrix_size=128x128, warmup=10, iters=100, avg_latency_ms=0.016625
kernel=torch, matrix_size=128x128, warmup=10, iters=100, avg_latency_ms=0.024258
kernel=naive, matrix_size=512x512, warmup=10, iters=100, avg_latency_ms=0.616350
kernel=tiled, matrix_size=512x512, warmup=10, iters=100, avg_latency_ms=0.728364
kernel=torch, matrix_size=512x512, warmup=10, iters=100, avg_latency_ms=0.108062
kernel=naive, matrix_size=1024x1024, warmup=10, iters=100, avg_latency_ms=4.798106
kernel=tiled, matrix_size=1024x1024, warmup=10, iters=100, avg_latency_ms=3.608767
kernel=torch, matrix_size=1024x1024, warmup=10, iters=100, avg_latency_ms=0.532411


## Observations

Benchmark Results

| Matrix Size | Naive CUDA | Tiled CUDA | PyTorch |
|---|---:|---:|---:|
| 128×128 | 0.025154 ms | **0.016625 ms** | 0.024258 ms |
| 512×512 | 0.616350 ms | 0.728364 ms | **0.108062 ms** |
| 1024×1024 | 4.798106 ms | 3.608767 ms | **0.532411 ms** |

**Observation 1 — Small Workloads Are Strongly Affected by Fixed Overheads**

At 128×128, all three implementations execute in only tens of microseconds:

- Naive CUDA: ~0.025 ms
- Tiled CUDA: ~0.017 ms
- PyTorch: ~0.024 ms

The differences at this size are extremely small.

For small workloads, fixed costs such as kernel launch overhead, synchronization, and other execution overheads become significant relative to the actual computation.

The tiled kernel happened to be the fastest in this measurement, but such tiny differences should not be overinterpreted.

### Takeaway

> For very small GPU workloads, fixed execution overhead can be comparable to the useful computation itself.

**Observation 2 — Tiling Is Not Automatically Faster**

At 512×512:

- Naive CUDA: 0.616350 ms
- Tiled CUDA: 0.728364 ms

The tiled implementation was approximately **18% slower** than the naive implementation.

This initially appears surprising because Experiment 4 showed that tiling dramatically reduced global-memory load requests.

However, reducing global-memory traffic does not automatically guarantee lower execution time.

The tiled kernel introduces additional work:

- Shared-memory loads and stores
- Two `__syncthreads()` barriers for every K tile
- Tile indexing and control logic
- Additional instructions associated with explicitly managing shared memory

At 512×512, the benefits from additional data reuse were apparently not sufficient to overcome these costs in this simple tiled implementation.

Observation 3 — Tiling Becomes Beneficial at the Larger Workload

At 1024×1024:

- Naive CUDA: 4.798106 ms
- Tiled CUDA: 3.608767 ms

The tiled implementation was approximately **25% faster** than the naive implementation.

As the matrix becomes larger, the amount of computation and data reuse increases substantially.

The cost of synchronization and shared-memory management becomes less significant relative to the amount of useful work performed.

The reuse enabled by tiling therefore becomes valuable enough to produce a clear latency improvement.

### Takeaway

> Optimization benefits depend on workload size. An optimization that loses at one problem size can become beneficial as the workload grows.

**Observation 4 — PyTorch Separates Dramatically as Workload Size Increases**

At 512×512:

- PyTorch was approximately **5.7× faster than Naive CUDA**
- PyTorch was approximately **6.7× faster than Tiled CUDA**

At 1024×1024:

- PyTorch was approximately **9.0× faster than Naive CUDA**
- PyTorch was approximately **6.8× faster than Tiled CUDA**

The performance difference becomes increasingly obvious as matrix size increases.

The custom tiled kernel demonstrates the basic principle of shared-memory data reuse, but it remains a relatively simple GEMM implementation.

PyTorch's GPU matrix multiplication uses a highly optimized underlying GEMM implementation and therefore goes substantially beyond simple 16×16 shared-memory tiling.

### Takeaway

> Shared-memory tiling is an important GPU optimization technique, but implementing tiling alone does not make a kernel comparable to a production-quality GEMM implementation.


**Observation 5 — Correct Synchronization Is Essential for CUDA Benchmarking**

CUDA kernel execution is asynchronous with respect to the CPU.

Therefore, code such as:

    start = time.perf_counter()
    C = kernel(A, B)
    end = time.perf_counter()

does not necessarily measure the complete GPU execution time.

The CPU may continue immediately after submitting the kernel while the GPU is still executing it.

The correct benchmark structure used in this experiment was:

    warm-up
        ↓
    torch.cuda.synchronize()
        ↓
    start timer
        ↓
    launch kernel repeatedly
        ↓
    torch.cuda.synchronize()
        ↓
    stop timer
        ↓
    divide total time by number of iterations

The synchronization before timing ensures that previous GPU work has completed.

The synchronization after the timed launches ensures that all benchmarked GPU work has actually completed before the timer is stopped.

Therefore, the measured duration represents the average completion time of the batch of GPU operations rather than merely CPU-side kernel submission time.


Overall Observations

1. Small GPU workloads can be heavily influenced by fixed execution and launch overhead.

2. Shared-memory tiling dramatically reducing memory activity does not guarantee an immediate latency improvement.

3. At 512×512, the simple tiled implementation was slower than naive CUDA despite its improved memory-reuse strategy.

4. At 1024×1024, the tiled implementation became approximately 25% faster than naive CUDA, showing that the benefits of reuse became more significant at larger workloads.

5. PyTorch substantially outperformed both custom implementations as workload size increased.

6. GPU optimization must ultimately be evaluated using end-to-end performance measurements rather than assuming that improvement in one hardware metric guarantees overall speedup.

7. CUDA's asynchronous execution model makes explicit synchronization essential when using CPU wall-clock timers for GPU benchmarking.

## Conclusions



> Correct GPU benchmarking requires understanding CUDA's asynchronous execution model. Warm-up iterations, synchronization, repeated measurements, and averaging are necessary to obtain meaningful latency measurements.

> The experiment also demonstrated that optimization is workload-dependent: shared-memory tiling can dramatically improve memory behavior without immediately improving latency, but its benefits become increasingly visible as the workload grows.